In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Ruta al archivo CSV
file_path = "Mediciones/Ring_Downs/ringdown-02-11-1655.csv"  # Reemplaza con el nombre de tu archivo

# 2. Definir tipos de datos optimizados para ahorrar memoria (float32 suele ser más que suficiente)
dtypes = {
    'tiempo': 'float32',
    'ch1_excitado': 'float32',
    'ch2_libre': 'float32'
}

# 3. Importar el CSV
# Ajusta 'sep' (e.g., ',' o ';') y 'decimal' (e.g., '.' o ',') según la estructura de tus datos.
# Si el CSV no tiene nombres de columnas en la primera línea, usa: header=None, names=['tiempo', 'ch1_excitado', 'ch2_libre']
df = pd.read_csv(
    file_path, 
    dtype=dtypes, 
    # sep=',',       # Descomenta si usas otro separador como ';'
    # decimal='.',   # Descomenta si usas ',' como coma decimal
)

# Renombrar columnas para trabajar cómodamente (ajusta los nombres si el CSV ya tiene encabezado)
df.columns = ['tiempo', 'v_excitada', 'v_libre']

# 4. Verificación rápida del dataset cargado
print(f"Líneas cargadas: {len(df):,}")
print(f"Memoria utilizada: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
df.head()

Líneas cargadas: 1,000,000
Memoria utilizada: 22.89 MB


,tiempo,v_excitada,v_libre
0,-0.200000,-0.01524,-0.066
1,-0.200000,-0.01524,-0.132
2,-0.199999,-0.01524,-0.204
3,-0.199999,-0.01524,-0.258
4,-0.199998,-0.01524,-0.324


In [ ]:
import sys
print(sys.executable)

In [10]:
import pandas as pd
import numpy as np
import os

# 1. Cargar el archivo original optimizando memoria
file_path = "Mediciones/Ring_Downs/ringdown-02-11-1655.csv"  # Reemplaza con tu archivo
dtypes = {'Tiempo (s)': 'float32', 'CH1 (V)': 'float32', 'CH2 (V)': 'float32'}
df = pd.read_csv(file_path, dtype=dtypes)

# Renombrar columnas para facilitar el manejo
df.columns = ['tiempo', 'ch1', 'ch2']

# 2. Acomodar el tiempo: desplazar para que empiece en 0 (pasa de aprox -200ms/200ms a 0/400ms)
t_min = df['tiempo'].min()
df['tiempo_ajustado'] = df['tiempo'] - t_min

# 3. Detección automática del punto de corte (Liberación de la excitación)
# Durante la excitación, CH2 tiene valores grandes (orden de 100 mV / 0.1 V o más).
# Buscamos el índice donde CH2 cae bruscamente hacia valores cercanos a 0 (oscilación libre).
# Usamos un umbral, por ejemplo, cuando CH2 baja de 0.05 V (50 mV) por primera vez sostenidamente.
umbral_corte = 0.05 

# Encontramos la primera fila donde CH2 cumple la condición después de estar alto
idx_corte = df[df['ch2'].abs() > umbral_corte].index[-1] 
# O alternativamente buscando el descenso:
# (Una forma robusta es buscar el punto de máxima caída de la envolvente de CH2)

print(f"Punto de corte detectado en el índice: {idx_corte}")
print(f"Tiempo ajustado del corte: {df.loc[idx_corte, 'tiempo_ajustado']:.6f} s")

# 4. Reconstrucción de la señal
# - Antes del corte (Excitación): Usamos CH2
# - Después del corte (Oscilación libre): Usamos CH1
df['senal_reconstruida'] = np.where(df.index <= idx_corte, df['ch2'], df['ch1'])

# 5. Crear el nuevo DataFrame final solo con el tiempo ajustado y la señal limpia
df_final = df[['tiempo_ajustado', 'senal_reconstruida']].copy()
df_final.columns = ['tiempo', 'senal']


# 6. Definir la ruta y crear la carpeta de destino si no existe
carpeta_destino = "Mediciones/Ring_Downs/Limpios"
os.makedirs(carpeta_destino, exist_ok=True)

# 7. Generar automáticamente el nombre del archivo de salida basado en el de entrada
# 'file_path' es la variable donde guardaste la ruta del archivo original (ej: "datos_osciloscopio.csv")
nombre_base = os.path.basename(file_path)                    # Extrae "datos_osciloscopio.csv"
nombre_sin_ext = os.path.splitext(nombre_base)[0]            # Extrae "datos_osciloscopio"
nombre_salida = f"{nombre_sin_ext}_reconstruida.csv"         # "datos_osciloscopio_reconstruida.csv"

# Unir la ruta de la carpeta con el nuevo nombre de archivo
ruta_salida = os.path.join(carpeta_destino, nombre_salida)

# 8. Guardar el archivo CSV en la ubicación especificada
df_final.to_csv(ruta_salida, index=False)

print(f"¡Proceso completado! Archivo guardado exitosamente en:\n{ruta_salida}")
df_final.to_csv(ruta_salida, index=False)

print(f"¡Proceso completado! Archivo guardado exitosamente como '{archivo_salida}'.")

print("¡Proceso completado! Archivo nuevo guardado exitosamente.")

Punto de corte detectado en el índice: 380182
Tiempo ajustado del corte: 0.152073 s
¡Proceso completado! Archivo guardado exitosamente en:
Mediciones/Ring_Downs/Limpios\ringdown-02-11-1655_reconstruida.csv
¡Proceso completado! Archivo guardado exitosamente como 'senal_reconstruida.csv'.
¡Proceso completado! Archivo nuevo guardado exitosamente.


In [ ]:
from scipy.signal import hilbert

# 1. Definir el tiempo de corte (asegúrate de que coincida con el valor donde se quita la excitación)
# Si en tu reescalado el corte ocurre en un tiempo específico, ponlo aquí:
t_corte = 0.0  # O el valor exacto del tiempo ajustado donde arranca la oscilación libre

# 2. Filtrar el DataFrame para quedarnos SOLO con la oscilación libre (desde t_corte en adelante)
df_libre = df_final[df_final['tiempo'] >= t_corte].copy()

tiempo_libre = df_libre['tiempo'].values
senal_libre = df_libre['senal'].values

# 3. Calcular la Transformada de Hilbert y la envolvente para la sección libre
senal_analitica = hilbert(senal_libre)
envolvente_libre = np.abs(senal_analitica)

# 4. Visualizar la señal libre junto con su envolvente
plt.figure(figsize=(10, 5))
plt.plot(tiempo_libre, senal_libre, label='Oscilación Libre', color='dodgerblue', alpha=0.6, lw=0.8)
plt.plot(tiempo_libre, envolvente_libre, label='Envolvente Superior', color='crimson', lw=1.5)
plt.plot(tiempo_libre, -envolvente_libre, label='Envolvente Inferior', color='crimson', lw=1.5, linestyle='--')

plt.xlabel('Tiempo (s)')
plt.ylabel('Tensión (V)')
plt.legend()
plt.title('Envolvente de la Oscilación Libre (Post-Corte)')
plt.grid(True)
plt.show()